# Image Classification with Neural Networks
## Skin Lesion Diagnosis & Facial Expression Recognition

Two multi-class classification tasks on imbalanced real-world datasets using PyTorch.
Both datasets have 7 classes and severe class imbalance, requiring careful handling.

| Dataset | Images | Classes | Imbalance ratio |
|---------|--------|---------|-----------------|
| Skin Moles (VAI) | 10,015 | 7 dermatological conditions | ~58:1 |
| Face Expressions (CANDID) | 15,339 | 7 emotions | ~17:1 |

Models trained: MLP, CNN (custom), ResNet-18 (fine-tuned), MobileNet-v2 (fine-tuned)

---
## Setup

In [ ]:
!pip install -q albumentations

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR, OneCycleLR

import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

# reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# dataset paths - Kaggle input
ROOT_DIR  = Path("/kaggle/input/tema2ia/Tema2_IA")

MOLES_DIR       = ROOT_DIR / "vai-de-pielea-mea"
MOLES_TRAIN_CSV = MOLES_DIR / "train.csv"
MOLES_TEST_CSV  = MOLES_DIR / "test.csv"
MOLES_KAG_CSV   = MOLES_DIR / "kaggle_test.csv"
MOLES_TRAIN_IMG = MOLES_DIR / "train"
MOLES_TEST_IMG  = MOLES_DIR / "test"
MOLES_KAG_IMG   = MOLES_DIR / "kaggle_test"

FACE_DIR        = ROOT_DIR / "you-re-on-candid-camera"
FACE_TRAIN_CSV  = FACE_DIR / "splits" / "train.csv"
FACE_LTEST_CSV  = FACE_DIR / "splits" / "local_test.csv"
FACE_RTEST_CSV  = FACE_DIR / "splits" / "remote_test.csv"
FACE_NORM_JSON  = FACE_DIR / "splits" / "norm_stats.json"
FACE_IMG_DIR    = FACE_DIR

# normalization stats computed on training set
MOLES_MEAN = [0.7646, 0.5464, 0.5712]
MOLES_STD  = [0.1410, 0.1532, 0.1706]
with open(FACE_NORM_JSON) as f:
    _ns = json.load(f)
FACE_MEAN = _ns["mean"]; FACE_STD = _ns["std"]

print("Moles mean:", MOLES_MEAN)
print("Faces mean:", FACE_MEAN)

---
## Datasets

In [ ]:
MOLES_CLASSES   = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
MOLES_CLASS2IDX = {c: i for i, c in enumerate(MOLES_CLASSES)}
IDX2CLASS_MOLES = {i: c for i, c in enumerate(MOLES_CLASSES)}

# CSV has columns: imagine, diagnostic
class MolesDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None, is_test=False):
        self.df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test
        self.labels = (None if is_test
                       else [MOLES_CLASS2IDX[d] for d in self.df["diagnostic"]])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]["imagine"]
        img = np.array(Image.open(self.img_dir / fname).convert("RGB"))
        if self.transform is not None:
            try:
                img = self.transform(image=img)["image"]
            except TypeError:
                img = self.transform(Image.fromarray(img))
        if self.is_test:
            return img, fname
        return img, self.labels[idx]

In [ ]:
FACE_CLASSES = ["Surprise", "Fear", "Disgust", "Happiness", "Sadness", "Anger", "Neutral"]

# CSV has columns: path (relative), label (1-7); test CSV has only: id
class FaceDataset(Dataset):
    def __init__(self, csv_path, base_dir, transform=None, is_test=False):
        self.df = pd.read_csv(csv_path)
        self.base_dir = Path(base_dir)
        self.transform = transform
        self.is_test = is_test
        if is_test:
            # test images are stored in class subfolders, build a name->path index
            test_dir = self.base_dir / "DATASET" / "test"
            self._index = {}
            for sub in test_dir.iterdir():
                if sub.is_dir():
                    for p in sub.iterdir():
                        self._index[p.name] = str(p)
            self.paths = [self._index[r["id"]] for _, r in self.df.iterrows()]
            self.labels = None
        else:
            self.paths = [str(self.base_dir / r["path"]) for _, r in self.df.iterrows()]
            self.labels = [int(r["label"]) - 1 for _, r in self.df.iterrows()]  # 0-indexed

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.paths[idx]).convert("RGB"))
        if self.transform is not None:
            try:
                img = self.transform(image=img)["image"]
            except TypeError:
                img = self.transform(Image.fromarray(img))
        if self.is_test:
            return img, self.df.iloc[idx]["id"]
        return img, self.labels[idx]

In [ ]:
# inverse-frequency weights for CrossEntropyLoss
def get_class_weights(labels, num_classes, device=DEVICE):
    counts = np.bincount(labels, minlength=num_classes).astype(float)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32).to(device)

# WeightedRandomSampler - oversamples rare classes so each batch is balanced
def get_sampler(labels):
    counts = np.bincount(labels)
    sample_w = (1.0 / counts)[labels]
    return WeightedRandomSampler(torch.from_numpy(sample_w).double(),
                                 num_samples=len(sample_w), replacement=True)

def plot_class_distribution(labels, class_names, title):
    counts = np.bincount(labels, minlength=len(class_names))
    fig, ax = plt.subplots(figsize=(11, 4))
    bars = ax.bar(class_names, counts, color=plt.cm.tab10.colors[:len(class_names)])
    ax.bar_label(bars); ax.set_title(title); ax.set_ylabel("Count")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

moles_tmp = MolesDataset(MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, transform=None)
face_tmp  = FaceDataset(FACE_TRAIN_CSV, FACE_IMG_DIR, transform=None)
plot_class_distribution(moles_tmp.labels, MOLES_CLASSES, "Class Distribution - Skin Moles (train)")
plot_class_distribution(face_tmp.labels, FACE_CLASSES, "Class Distribution - Face Expressions (train)")

---
## Data Augmentation

**Skin Moles:** lesions have no preferred orientation so flips and 90-degree rotations are valid.
Color jitter is kept mild (±10%) to avoid corrupting diagnostic color cues.

**Face Expressions:** horizontal flip is valid (faces are symmetric). No vertical flip -
upside-down faces don't appear in real data. Rotation limited to ±15 degrees.

In [ ]:
IMG_SIZE = 96  # resize target for MLP and CNN

moles_aug_train = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=MOLES_MEAN, std=MOLES_STD), ToTensorV2()])

moles_aug_val = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MOLES_MEAN, std=MOLES_STD), ToTensorV2()])

face_aug_train = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5), A.Rotate(limit=15, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
    A.Normalize(mean=FACE_MEAN, std=FACE_STD), ToTensorV2()])

face_aug_val = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=FACE_MEAN, std=FACE_STD), ToTensorV2()])

print("Augmentation pipelines ready.")

In [ ]:
# visualize original + 5 augmented versions for each class
def show_augmentations(dataset, aug_transform, class_names, mean, std, n_aug=5, title=""):
    first_idx = {}
    for i, lbl in enumerate(dataset.labels):
        if lbl not in first_idx:
            first_idx[lbl] = i
        if len(first_idx) == len(class_names):
            break

    fig, axes = plt.subplots(len(class_names), n_aug + 1,
                              figsize=(2.2 * (n_aug + 1), 2.2 * len(class_names)))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    for row, cls_idx in enumerate(sorted(first_idx)):
        idx = first_idx[cls_idx]
        img_path = (dataset.img_dir / dataset.df.iloc[idx]["imagine"]
                    if hasattr(dataset, "img_dir") else dataset.paths[idx])
        img_orig = np.array(Image.open(img_path).convert("RGB"))

        axes[row, 0].imshow(img_orig.astype(float) / 255.0)
        axes[row, 0].set_title("original", fontsize=8)
        axes[row, 0].set_ylabel(class_names[cls_idx], fontsize=9, rotation=0, labelpad=45, va="center")
        axes[row, 0].axis("off")

        for col in range(1, n_aug + 1):
            aug = aug_transform(image=img_orig)["image"]
            disp = np.clip(aug.permute(1,2,0).numpy() * np.array(std) + np.array(mean), 0, 1)
            axes[row, col].imshow(disp)
            axes[row, col].set_title(f"aug {col}", fontsize=8)
            axes[row, col].axis("off")

    plt.tight_layout(); plt.show()

moles_raw = MolesDataset(MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, transform=None)
face_raw  = FaceDataset(FACE_TRAIN_CSV, FACE_IMG_DIR, transform=None)
show_augmentations(moles_raw, moles_aug_train, MOLES_CLASSES, MOLES_MEAN, MOLES_STD, 5, "Augmentations - Skin Moles")
show_augmentations(face_raw, face_aug_train, FACE_CLASSES, FACE_MEAN, FACE_STD, 5, "Augmentations - Face Expressions")

---
## Model Architectures

### MLP
Three blocks of Linear -> BatchNorm1d -> ReLU -> Dropout with decreasing dropout rates (0.4 -> 0.3 -> 0.2).
BatchNorm stabilizes training; dropout rates decrease toward the output since later layers have fewer parameters.

### CNN
Three conv blocks (32 -> 64 -> 128 channels) with MaxPool, followed by AdaptiveAvgPool and two linear layers.
First kernel is 5x5 for skin moles (coarser texture features) and 3x3 for faces (fine local features like eyes/mouth).
`use_bn` and `use_dropout` flags enable ablation studies without rewriting the class.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, num_classes, use_dropout=True):
        super().__init__()
        drop = lambda p: nn.Dropout(p) if use_dropout else nn.Identity()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, 1024), nn.BatchNorm1d(1024), nn.ReLU(inplace=True), drop(0.4),
            nn.Linear(1024, 512),  nn.BatchNorm1d(512),  nn.ReLU(inplace=True), drop(0.3),
            nn.Linear(512, 256),   nn.BatchNorm1d(256),  nn.ReLU(inplace=True), drop(0.2),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.net(x)

def _conv_block(in_ch, out_ch, kernel_size, use_bn=True):
    # bias=False when followed by BN since BN has its own beta
    layers = [nn.Conv2d(in_ch, out_ch, kernel_size, padding=kernel_size // 2, bias=not use_bn)]
    if use_bn:
        layers.append(nn.BatchNorm2d(out_ch))
    layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

class CNN(nn.Module):
    def __init__(self, num_classes=7, first_kernel=3, use_bn=True):
        super().__init__()
        self.features = nn.Sequential(
            _conv_block(3,   32,  first_kernel, use_bn), nn.MaxPool2d(2),
            _conv_block(32,  64,  3,            use_bn), nn.MaxPool2d(2),
            _conv_block(64,  128, 3,            use_bn), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),  # makes model resolution-agnostic
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(128, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

input_size = 3 * IMG_SIZE * IMG_SIZE
_x = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
print("MLP params:", sum(p.numel() for p in MLP(input_size, 7).parameters()))
print("CNN params:", sum(p.numel() for p in CNN(7, first_kernel=5).parameters()))

---
## Training Utilities

Key design choices:
- **Label smoothing (0.1):** reduces overconfidence, improves generalization
- **OneCycleLR:** built-in warmup + cosine annealing, faster convergence in fewer epochs
- **Best model selection by F1-macro** (not val loss): matches the Kaggle evaluation metric and is more sensitive to rare classes

In [ ]:
BATCH_SIZE   = 64
NUM_EPOCHS   = 20
LABEL_SMOOTH = 0.1

def train_one_epoch(model, loader, criterion, optimizer, scheduler=None, device=DEVICE):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward(); optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (model(imgs).argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device=DEVICE):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                scheduler=None, batch_scheduler=False, num_epochs=NUM_EPOCHS,
                run_name="run", device=DEVICE, verbose=True):
    writer = SummaryWriter(log_dir=f"/kaggle/working/runs/{run_name}")
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
    best_f1, best_state = -1.0, None

    for epoch in range(num_epochs):
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, criterion, optimizer,
            scheduler if batch_scheduler else None, device)
        va_loss, va_acc, preds, labels = evaluate(model, val_loader, criterion, device)
        va_f1 = f1_score(labels, preds, average="macro", zero_division=0)

        if scheduler is not None and not batch_scheduler:
            scheduler.step()

        for k, v in zip(["train_loss","train_acc","val_loss","val_acc","val_f1"],
                        [tr_loss, tr_acc, va_loss, va_acc, va_f1]):
            history[k].append(v)
            writer.add_scalar(k, v, epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

        if va_f1 > best_f1:
            best_f1 = va_f1; best_state = deepcopy(model.state_dict())

        if verbose and (epoch + 1) % 5 == 0:
            print(f"  Ep {epoch+1:02d}/{num_epochs} | "
                  f"train loss={tr_loss:.3f} acc={tr_acc:.3f} | "
                  f"val loss={va_loss:.3f} acc={va_acc:.3f} f1={va_f1:.3f}")

    writer.close()
    model.load_state_dict(best_state)
    print(f"  Best val F1-macro: {best_f1:.4f}")
    return history

In [ ]:
def plot_ablation_curves(hist_a, hist_b, label_a, label_b, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for h, lbl, ls in [(hist_a, label_a, "-"), (hist_b, label_b, "--")]:
        axes[0].plot(h["train_loss"], ls=ls, label=f"{lbl} train")
        axes[0].plot(h["val_loss"],   ls=ls, alpha=0.6, label=f"{lbl} val")
        axes[1].plot(h["train_acc"],  ls=ls, label=f"{lbl} train")
        axes[1].plot(h["val_acc"],    ls=ls, alpha=0.6, label=f"{lbl} val")
        axes[2].plot(h["val_f1"],     ls=ls, label=f"{lbl} val F1")
    for ax, t in zip(axes, ["Loss", "Accuracy", "F1-macro"]):
        ax.set_title(t); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.suptitle(title, fontweight="bold"); plt.tight_layout(); plt.show()

def plot_multi_curves(histories, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for i, (name, h) in enumerate(histories.items()):
        axes[0].plot(h["val_loss"], color=plt.cm.tab10.colors[i], label=name)
        axes[1].plot(h["val_f1"],   color=plt.cm.tab10.colors[i], label=name)
    for ax, t in zip(axes, ["Val Loss", "Val F1-macro"]):
        ax.set_title(t); ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle(title, fontweight="bold"); plt.tight_layout(); plt.show()

def print_metrics(all_preds, all_labels, class_names, title=""):
    print(f"\n{'='*60}\n {title}\n{'='*60}")
    acc    = accuracy_score(all_labels, all_preds)
    f1_mac = f1_score(all_labels, all_preds, average="macro",  zero_division=0)
    f1_mic = f1_score(all_labels, all_preds, average="micro",  zero_division=0)
    print(f"Accuracy : {acc:.4f}\nF1-macro : {f1_mac:.4f}\nF1-micro : {f1_mic:.4f}\n")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, cm[i,j], ha="center", va="center",
                    color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=8)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix - {title}")
    plt.colorbar(im); plt.tight_layout(); plt.show()
    return {"acc": acc, "f1_macro": f1_mac, "f1_micro": f1_mic}

---
## MLP - Training & Dropout Ablation

In [ ]:
# build dataloaders - sampler handles class imbalance at batch level
moles_ds_train = MolesDataset(MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, moles_aug_train)
moles_ds_val   = MolesDataset(MOLES_TEST_CSV,  MOLES_TEST_IMG,  moles_aug_val)
face_ds_train  = FaceDataset(FACE_TRAIN_CSV, FACE_IMG_DIR, face_aug_train)
face_ds_val    = FaceDataset(FACE_LTEST_CSV, FACE_IMG_DIR, face_aug_val)

moles_train_loader = DataLoader(moles_ds_train, batch_size=BATCH_SIZE,
                                sampler=get_sampler(np.array(moles_ds_train.labels)),
                                num_workers=2, drop_last=True)
moles_val_loader   = DataLoader(moles_ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
face_train_loader  = DataLoader(face_ds_train, batch_size=BATCH_SIZE,
                                sampler=get_sampler(np.array(face_ds_train.labels)),
                                num_workers=2, drop_last=True)
face_val_loader    = DataLoader(face_ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

crit_eval = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
print(f"Moles: {len(moles_ds_train)} train / {len(moles_ds_val)} val")
print(f"Faces: {len(face_ds_train)} train / {len(face_ds_val)} val")

In [ ]:
def train_mlp(use_dropout, train_loader, val_loader, run_name):
    model = MLP(3 * IMG_SIZE * IMG_SIZE, num_classes=7, use_dropout=use_dropout).to(DEVICE)
    crit  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
    sched = OneCycleLR(opt, max_lr=2e-3, epochs=NUM_EPOCHS, steps_per_epoch=len(train_loader))
    return model, train_model(model, train_loader, val_loader, crit, opt,
                              scheduler=sched, batch_scheduler=True,
                              num_epochs=NUM_EPOCHS, run_name=run_name)

print("[Moles] MLP with Dropout")
mlp_moles_drop, hist_mlp_moles_drop = train_mlp(True, moles_train_loader, moles_val_loader, "mlp_moles_drop")
_, _, p, l = evaluate(mlp_moles_drop, moles_val_loader, crit_eval)
met_mlp_moles_drop = print_metrics(p, l, MOLES_CLASSES, "MLP (Dropout) - Skin Moles")

print("\n[Moles] MLP without Dropout")
mlp_moles_nodrop, hist_mlp_moles_nodrop = train_mlp(False, moles_train_loader, moles_val_loader, "mlp_moles_nodrop")
_, _, p, l = evaluate(mlp_moles_nodrop, moles_val_loader, crit_eval)
met_mlp_moles_nodrop = print_metrics(p, l, MOLES_CLASSES, "MLP (no Dropout) - Skin Moles")

In [ ]:
print("[Faces] MLP with Dropout")
mlp_face_drop, hist_mlp_face_drop = train_mlp(True, face_train_loader, face_val_loader, "mlp_face_drop")
_, _, p, l = evaluate(mlp_face_drop, face_val_loader, crit_eval)
met_mlp_face_drop = print_metrics(p, l, FACE_CLASSES, "MLP (Dropout) - Face Expressions")

print("\n[Faces] MLP without Dropout")
mlp_face_nodrop, hist_mlp_face_nodrop = train_mlp(False, face_train_loader, face_val_loader, "mlp_face_nodrop")
_, _, p, l = evaluate(mlp_face_nodrop, face_val_loader, crit_eval)
met_mlp_face_nodrop = print_metrics(p, l, FACE_CLASSES, "MLP (no Dropout) - Face Expressions")

In [ ]:
plot_ablation_curves(hist_mlp_moles_drop, hist_mlp_moles_nodrop,
                     "with Dropout", "no Dropout", "Dropout Ablation - MLP Skin Moles")
plot_ablation_curves(hist_mlp_face_drop, hist_mlp_face_nodrop,
                     "with Dropout", "no Dropout", "Dropout Ablation - MLP Face Expressions")

---
## CNN - BatchNorm & Augmentation Ablations

In [ ]:
# 4 augmentation configs: none, geometric only, color only, all
def make_moles_aug(geom=False, color=False):
    steps = [A.Resize(IMG_SIZE, IMG_SIZE)]
    if geom:
        steps += [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
                  A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.5)]
    if color:
        steps += [A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02, p=0.5),
                  A.GaussianBlur(blur_limit=(3, 5), p=0.3)]
    steps += [A.Normalize(mean=MOLES_MEAN, std=MOLES_STD), ToTensorV2()]
    return A.Compose(steps)

def make_face_aug(geom=False, color=False):
    steps = [A.Resize(IMG_SIZE, IMG_SIZE)]
    if geom:
        steps += [A.HorizontalFlip(p=0.5), A.Rotate(limit=15, p=0.5),
                  A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4)]
    if color:
        steps += [A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
                  A.HueSaturationValue(10, 20, 10, p=0.4), A.GaussianBlur(blur_limit=(3,5), p=0.3)]
    steps += [A.Normalize(mean=FACE_MEAN, std=FACE_STD), ToTensorV2()]
    return A.Compose(steps)

aug_configs = {
    "none":     (make_moles_aug(False, False), make_face_aug(False, False)),
    "geom":     (make_moles_aug(True,  False), make_face_aug(True,  False)),
    "color":    (make_moles_aug(False, True),  make_face_aug(False, True)),
    "geom+color":(make_moles_aug(True, True),  make_face_aug(True,  True)),
}

In [ ]:
def train_cnn(dataset_name, aug_train, aug_val, csv_train, img_train, csv_val, img_val,
              first_kernel=3, use_bn=True, use_sampler=True, use_wloss=False, run_suffix=""):
    is_moles = dataset_name == "moles"
    DS = MolesDataset if is_moles else FaceDataset
    ds_train = DS(csv_train, img_train, aug_train)
    ds_val   = DS(csv_val,   img_val,   aug_val)

    loader_tr = DataLoader(ds_train, batch_size=BATCH_SIZE, num_workers=2, drop_last=True,
                           sampler=get_sampler(np.array(ds_train.labels)) if use_sampler else None,
                           shuffle=not use_sampler)
    loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    cw = get_class_weights(np.array(ds_train.labels), 7) if use_wloss else None
    model = CNN(num_classes=7, first_kernel=first_kernel, use_bn=use_bn).to(DEVICE)
    crit  = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTH)
    opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
    sched = OneCycleLR(opt, max_lr=3e-3, epochs=NUM_EPOCHS, steps_per_epoch=len(loader_tr))

    hist = train_model(model, loader_tr, loader_val, crit, opt,
                       scheduler=sched, batch_scheduler=True,
                       num_epochs=NUM_EPOCHS, run_name=f"cnn_{dataset_name}_{run_suffix}",
                       verbose=False)
    _, _, preds, labels = evaluate(model, loader_val, crit)
    metrics = print_metrics(preds, labels, MOLES_CLASSES if is_moles else FACE_CLASSES,
                            f"CNN {dataset_name} - {run_suffix}")
    return model, hist, metrics

In [ ]:
# BatchNorm ablation - same augmentation, toggle BN
print("=== BatchNorm Ablation - Skin Moles ===")
cnn_moles_bn, hist_cnn_moles_bn, met_cnn_moles_bn = train_cnn(
    "moles", moles_aug_train, moles_aug_val,
    MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, MOLES_TEST_CSV, MOLES_TEST_IMG,
    first_kernel=5, use_bn=True, run_suffix="bn")
_, hist_cnn_moles_nobn, met_cnn_moles_nobn = train_cnn(
    "moles", moles_aug_train, moles_aug_val,
    MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, MOLES_TEST_CSV, MOLES_TEST_IMG,
    first_kernel=5, use_bn=False, run_suffix="no_bn")
plot_ablation_curves(hist_cnn_moles_bn, hist_cnn_moles_nobn, "with BN", "no BN",
                     "BatchNorm Ablation - CNN Skin Moles")

In [ ]:
print("=== BatchNorm Ablation - Face Expressions ===")
cnn_face_bn, hist_cnn_face_bn, met_cnn_face_bn = train_cnn(
    "face", face_aug_train, face_aug_val,
    FACE_TRAIN_CSV, FACE_IMG_DIR, FACE_LTEST_CSV, FACE_IMG_DIR,
    first_kernel=3, use_bn=True, run_suffix="bn")
_, hist_cnn_face_nobn, met_cnn_face_nobn = train_cnn(
    "face", face_aug_train, face_aug_val,
    FACE_TRAIN_CSV, FACE_IMG_DIR, FACE_LTEST_CSV, FACE_IMG_DIR,
    first_kernel=3, use_bn=False, run_suffix="no_bn")
plot_ablation_curves(hist_cnn_face_bn, hist_cnn_face_nobn, "with BN", "no BN",
                     "BatchNorm Ablation - CNN Face Expressions")

In [ ]:
# augmentation ablation - 4 configs, BN always on
aug_hist_moles, aug_met_moles = {}, {}
for name, (m_aug, _) in aug_configs.items():
    print(f"\n=== Augmentation: {name} - Moles ===")
    _, h, m = train_cnn("moles", m_aug, moles_aug_val,
                        MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, MOLES_TEST_CSV, MOLES_TEST_IMG,
                        first_kernel=5, use_bn=True, run_suffix=f"aug_{name}")
    aug_hist_moles[name] = h; aug_met_moles[name] = m

aug_hist_face, aug_met_face = {}, {}
for name, (_, f_aug) in aug_configs.items():
    print(f"\n=== Augmentation: {name} - Faces ===")
    _, h, m = train_cnn("face", f_aug, face_aug_val,
                        FACE_TRAIN_CSV, FACE_IMG_DIR, FACE_LTEST_CSV, FACE_IMG_DIR,
                        first_kernel=3, use_bn=True, run_suffix=f"aug_{name}")
    aug_hist_face[name] = h; aug_met_face[name] = m

plot_multi_curves(aug_hist_moles, "Augmentation Ablation - CNN Skin Moles")
plot_multi_curves(aug_hist_face,  "Augmentation Ablation - CNN Face Expressions")

for title, metrics in [("Moles", aug_met_moles), ("Faces", aug_met_face)]:
    df = pd.DataFrame(metrics).T[["acc","f1_macro","f1_micro"]]
    df.columns = ["Accuracy","F1-macro","F1-micro"]
    print(f"\n{title}"); print(df.to_string())

---
## Class Imbalance Ablation

Comparing 4 strategies: no balancing, weighted loss only, sampler only, both combined.
Key finding: sampler + weighted loss together over-compensates - the model starts predicting
rare classes excessively, collapsing F1-macro to near-random levels.

In [ ]:
def train_imbalance_ablation(dataset_name, csv_train, img_train, csv_val, img_val,
                             aug_train, aug_val, first_kernel=3):
    results = {}
    for cfg, use_s, use_w in [("none", False, False), ("wloss", False, True),
                               ("sampler", True, False), ("sampler+wloss", True, True)]:
        print(f"  [{dataset_name}] {cfg}")
        _, h, m = train_cnn(dataset_name, aug_train, aug_val, csv_train, img_train, csv_val, img_val,
                            first_kernel=first_kernel, use_bn=True,
                            use_sampler=use_s, use_wloss=use_w, run_suffix=f"imb_{cfg}")
        results[cfg] = {"history": h, "metrics": m}
    return results

print("=== Imbalance Ablation - Skin Moles ===")
imb_moles = train_imbalance_ablation("moles", MOLES_TRAIN_CSV, MOLES_TRAIN_IMG,
                                     MOLES_TEST_CSV, MOLES_TEST_IMG,
                                     moles_aug_train, moles_aug_val, first_kernel=5)

print("\n=== Imbalance Ablation - Face Expressions ===")
imb_face = train_imbalance_ablation("face", FACE_TRAIN_CSV, FACE_IMG_DIR,
                                    FACE_LTEST_CSV, FACE_IMG_DIR,
                                    face_aug_train, face_aug_val, first_kernel=3)

In [ ]:
def report_imbalance(results, title):
    plot_multi_curves({k: v["history"] for k, v in results.items()}, title)
    df = pd.DataFrame({k: v["metrics"] for k, v in results.items()}).T[["acc","f1_macro","f1_micro"]]
    df.columns = ["Accuracy","F1-macro","F1-micro"]
    print(f"\n{title}"); print(df.to_string())

report_imbalance(imb_moles, "Imbalance Ablation - Skin Moles")
report_imbalance(imb_face,  "Imbalance Ablation - Face Expressions")

---
## Fine-Tuning - ResNet-18 & MobileNet-v2

Two strategies:
1. **Frozen backbone** (warmup ablation): only the classification head is trained
2. **Full fine-tuning**: entire network, with differential learning rates (1e-4 backbone, 5e-4 head)

Full fine-tuning consistently outperforms frozen by a large margin on both datasets.

In [ ]:
IMG_SIZE_FT   = 224  # pretrained models expect 224x224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

moles_ft_train = A.Compose([A.Resize(IMG_SIZE_FT, IMG_SIZE_FT),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02, p=0.4),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
moles_ft_val = A.Compose([A.Resize(IMG_SIZE_FT, IMG_SIZE_FT),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

face_ft_train = A.Compose([A.Resize(IMG_SIZE_FT, IMG_SIZE_FT),
    A.HorizontalFlip(p=0.5), A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])
face_ft_val = A.Compose([A.Resize(IMG_SIZE_FT, IMG_SIZE_FT),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])

In [ ]:
def build_resnet18(num_classes, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_mobilenet_v2(num_classes, freeze_backbone=True):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters(): p.requires_grad = False
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    return model

In [ ]:
WARMUP_EPOCHS = 3
NUM_EPOCHS_FT = 20

# frozen backbone - only head trains, used for warmup ablation
def finetune_frozen(model_builder, dataset_name, csv_train, img_train, csv_val, img_val,
                    aug_train, aug_val, use_warmup=True, run_suffix=""):
    is_moles = dataset_name == "moles"
    DS = MolesDataset if is_moles else FaceDataset
    ds_train = DS(csv_train, img_train, aug_train)
    ds_val   = DS(csv_val,   img_val,   aug_val)
    loader_tr  = DataLoader(ds_train, batch_size=32,
                            sampler=get_sampler(np.array(ds_train.labels)),
                            num_workers=2, drop_last=True)
    loader_val = DataLoader(ds_val, batch_size=32, shuffle=False, num_workers=2)

    model = model_builder(num_classes=7, freeze_backbone=True).to(DEVICE)
    crit  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    opt   = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=1e-3, weight_decay=1e-4)

    if use_warmup:
        sched = SequentialLR(opt,
                             schedulers=[LambdaLR(opt, lambda e: (e+1)/WARMUP_EPOCHS),
                                         CosineAnnealingLR(opt, T_max=NUM_EPOCHS_FT-WARMUP_EPOCHS)],
                             milestones=[WARMUP_EPOCHS])
    else:
        sched = CosineAnnealingLR(opt, T_max=NUM_EPOCHS_FT)

    tag = "warmup" if use_warmup else "no_warmup"
    hist = train_model(model, loader_tr, loader_val, crit, opt,
                       scheduler=sched, batch_scheduler=False,
                       num_epochs=NUM_EPOCHS_FT, run_name=f"ft_frozen_{dataset_name}_{run_suffix}_{tag}",
                       verbose=False)
    _, _, preds, labels = evaluate(model, loader_val, crit)
    metrics = print_metrics(preds, labels, MOLES_CLASSES if is_moles else FACE_CLASSES,
                            f"FT frozen {dataset_name} {run_suffix} {tag}")
    return model, hist, metrics

In [ ]:
# full fine-tuning with differential learning rates
def finetune_full(model_builder, dataset_name, csv_train, img_train, csv_val, img_val,
                  aug_train, aug_val, num_epochs=35, run_suffix=""):
    is_moles = dataset_name == "moles"
    DS = MolesDataset if is_moles else FaceDataset
    ds_train = DS(csv_train, img_train, aug_train)
    ds_val   = DS(csv_val,   img_val,   aug_val)
    loader_tr  = DataLoader(ds_train, batch_size=32,
                            sampler=get_sampler(np.array(ds_train.labels)),
                            num_workers=2, drop_last=True)
    loader_val = DataLoader(ds_val, batch_size=32, shuffle=False, num_workers=2)

    model = model_builder(num_classes=7, freeze_backbone=False).to(DEVICE)

    # lower LR for backbone to preserve ImageNet features
    backbone_params = [p for n, p in model.named_parameters() if "fc" not in n and "classifier" not in n]
    head_params     = [p for n, p in model.named_parameters() if "fc" in n or "classifier" in n]
    opt = optim.AdamW([{"params": backbone_params, "lr": 1e-4},
                       {"params": head_params,     "lr": 5e-4}], weight_decay=1e-4)

    crit  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    sched = OneCycleLR(opt, max_lr=[1e-4, 5e-4], epochs=num_epochs, steps_per_epoch=len(loader_tr))

    hist = train_model(model, loader_tr, loader_val, crit, opt,
                       scheduler=sched, batch_scheduler=True,
                       num_epochs=num_epochs, run_name=f"ft_full_{dataset_name}_{run_suffix}",
                       verbose=True)
    _, _, preds, labels = evaluate(model, loader_val, crit)
    metrics = print_metrics(preds, labels, MOLES_CLASSES if is_moles else FACE_CLASSES,
                            f"FT full {dataset_name} {run_suffix}")
    return model, hist, metrics

In [ ]:
# warmup ablation - frozen backbone
print("=== Warmup Ablation - ResNet-18 Skin Moles ===")
ft_moles_warm, hist_ft_moles_warm, met_ft_moles_warm = finetune_frozen(
    build_resnet18, "moles", MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, MOLES_TEST_CSV, MOLES_TEST_IMG,
    moles_ft_train, moles_ft_val, use_warmup=True, run_suffix="resnet18")
_, hist_ft_moles_nowarm, met_ft_moles_nowarm = finetune_frozen(
    build_resnet18, "moles", MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, MOLES_TEST_CSV, MOLES_TEST_IMG,
    moles_ft_train, moles_ft_val, use_warmup=False, run_suffix="resnet18")
plot_ablation_curves(hist_ft_moles_warm, hist_ft_moles_nowarm, "warmup", "no warmup",
                     "LR Warmup Ablation - ResNet-18 Skin Moles")

In [ ]:
print("=== Warmup Ablation - ResNet-18 Face Expressions ===")
ft_face_warm, hist_ft_face_warm, met_ft_face_warm = finetune_frozen(
    build_resnet18, "face", FACE_TRAIN_CSV, FACE_IMG_DIR, FACE_LTEST_CSV, FACE_IMG_DIR,
    face_ft_train, face_ft_val, use_warmup=True, run_suffix="resnet18")
_, hist_ft_face_nowarm, met_ft_face_nowarm = finetune_frozen(
    build_resnet18, "face", FACE_TRAIN_CSV, FACE_IMG_DIR, FACE_LTEST_CSV, FACE_IMG_DIR,
    face_ft_train, face_ft_val, use_warmup=False, run_suffix="resnet18")
plot_ablation_curves(hist_ft_face_warm, hist_ft_face_nowarm, "warmup", "no warmup",
                     "LR Warmup Ablation - ResNet-18 Face Expressions")

In [ ]:
# full fine-tuning - both ResNet-18 and MobileNet-v2
print("=== Full Fine-Tuning - Skin Moles ===")
ft_moles_resnet, hist_ft_moles_resnet, met_ft_moles_resnet = finetune_full(
    build_resnet18, "moles", MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, MOLES_TEST_CSV, MOLES_TEST_IMG,
    moles_ft_train, moles_ft_val, num_epochs=35, run_suffix="resnet18")

ft_moles_mobilenet, hist_ft_moles_mobilenet, met_ft_moles_mobilenet = finetune_full(
    build_mobilenet_v2, "moles", MOLES_TRAIN_CSV, MOLES_TRAIN_IMG, MOLES_TEST_CSV, MOLES_TEST_IMG,
    moles_ft_train, moles_ft_val, num_epochs=35, run_suffix="mobilenet_v2")

In [ ]:
print("=== Full Fine-Tuning - Face Expressions ===")
ft_face_resnet, hist_ft_face_resnet, met_ft_face_resnet = finetune_full(
    build_resnet18, "face", FACE_TRAIN_CSV, FACE_IMG_DIR, FACE_LTEST_CSV, FACE_IMG_DIR,
    face_ft_train, face_ft_val, num_epochs=35, run_suffix="resnet18")

ft_face_mobilenet, hist_ft_face_mobilenet, met_ft_face_mobilenet = finetune_full(
    build_mobilenet_v2, "face", FACE_TRAIN_CSV, FACE_IMG_DIR, FACE_LTEST_CSV, FACE_IMG_DIR,
    face_ft_train, face_ft_val, num_epochs=35, run_suffix="mobilenet_v2")

# save best models
import torch
torch.save(ft_moles_resnet.state_dict(),    "/kaggle/working/ft_moles_resnet18.pth")
torch.save(ft_moles_mobilenet.state_dict(), "/kaggle/working/ft_moles_mobilenet.pth")
torch.save(ft_face_resnet.state_dict(),     "/kaggle/working/ft_face_resnet18.pth")
torch.save(ft_face_mobilenet.state_dict(),  "/kaggle/working/ft_face_mobilenet.pth")
print("Models saved.")

---
## Predictions

In [ ]:
# pick best model per dataset based on val F1-macro
best_moles = (ft_moles_resnet
              if met_ft_moles_resnet["f1_macro"] >= met_ft_moles_mobilenet["f1_macro"]
              else ft_moles_mobilenet)
best_face  = (ft_face_resnet
              if met_ft_face_resnet["f1_macro"] >= met_ft_face_mobilenet["f1_macro"]
              else ft_face_mobilenet)

print(f"Best Moles: {'ResNet-18' if best_moles is ft_moles_resnet else 'MobileNet-v2'} "
      f"(F1: {max(met_ft_moles_resnet['f1_macro'], met_ft_moles_mobilenet['f1_macro']):.4f})")
print(f"Best Faces: {'ResNet-18' if best_face is ft_face_resnet else 'MobileNet-v2'} "
      f"(F1: {max(met_ft_face_resnet['f1_macro'], met_ft_face_mobilenet['f1_macro']):.4f})")

In [ ]:
@torch.no_grad()
def predict_moles(model, aug_val, out_csv="/kaggle/working/submission_moles.csv"):
    # submission format: Id (filename), Prediction (numeric class index 0-6)
    model.eval()
    ds = MolesDataset(MOLES_KAG_CSV, MOLES_KAG_IMG, aug_val, is_test=True)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    ids, preds = [], []
    for imgs, fnames in loader:
        ids.extend(fnames)
        preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy().tolist())
    df = pd.DataFrame({"Id": ids, "Prediction": preds})
    df.to_csv(out_csv, index=False)
    print(f"Saved {out_csv} ({len(df)} rows)")
    return df

@torch.no_grad()
def predict_faces(model, aug_val, out_csv="/kaggle/working/submission_faces.csv"):
    # submission format: id (filename), label (1-7)
    model.eval()
    test_dir = FACE_DIR / "DATASET" / "test"
    test_index = {p.name: str(p) for sub in test_dir.iterdir()
                  if sub.is_dir() for p in sub.iterdir()}
    df_csv = pd.read_csv(FACE_RTEST_CSV)
    ids, labels = [], []
    for _, row in df_csv.iterrows():
        img = np.array(Image.open(test_index[row["id"]]).convert("RGB"))
        img_t = aug_val(image=img)["image"].unsqueeze(0).to(DEVICE)
        ids.append(row["id"])
        labels.append(model(img_t).argmax(1).item() + 1)  # back to 1-indexed
    df = pd.DataFrame({"id": ids, "label": labels})
    df.to_csv(out_csv, index=False)
    print(f"Saved {out_csv} ({len(df)} rows)")
    return df

predict_moles(best_moles, moles_ft_val)
predict_faces(best_face,  face_ft_val)

# verify outputs
for fname in ["submission_moles.csv", "submission_faces.csv"]:
    df = pd.read_csv(f"/kaggle/working/{fname}")
    print(f"\n{fname}: {len(df)} rows, columns: {df.columns.tolist()}")
    print(df.head(3))

---
## Results Summary

In [ ]:
summary = {
    "Model": [
        "MLP (Dropout) - Moles",      "MLP (no Dropout) - Moles",
        "CNN (BN) - Moles",           "CNN (no BN) - Moles",
        "ResNet-18 frozen (warmup) - Moles", "ResNet-18 frozen (no warmup) - Moles",
        "ResNet-18 full FT - Moles",  "MobileNet-v2 full FT - Moles",
        "MLP (Dropout) - Faces",      "MLP (no Dropout) - Faces",
        "CNN (BN) - Faces",           "CNN (no BN) - Faces",
        "ResNet-18 frozen (warmup) - Faces", "ResNet-18 frozen (no warmup) - Faces",
        "ResNet-18 full FT - Faces",  "MobileNet-v2 full FT - Faces",
    ],
    "Accuracy": [
        met_mlp_moles_drop["acc"],      met_mlp_moles_nodrop["acc"],
        met_cnn_moles_bn["acc"],        met_cnn_moles_nobn["acc"],
        met_ft_moles_warm["acc"],       met_ft_moles_nowarm["acc"],
        met_ft_moles_resnet["acc"],     met_ft_moles_mobilenet["acc"],
        met_mlp_face_drop["acc"],       met_mlp_face_nodrop["acc"],
        met_cnn_face_bn["acc"],         met_cnn_face_nobn["acc"],
        met_ft_face_warm["acc"],        met_ft_face_nowarm["acc"],
        met_ft_face_resnet["acc"],      met_ft_face_mobilenet["acc"],
    ],
    "F1-macro": [
        met_mlp_moles_drop["f1_macro"],      met_mlp_moles_nodrop["f1_macro"],
        met_cnn_moles_bn["f1_macro"],        met_cnn_moles_nobn["f1_macro"],
        met_ft_moles_warm["f1_macro"],       met_ft_moles_nowarm["f1_macro"],
        met_ft_moles_resnet["f1_macro"],     met_ft_moles_mobilenet["f1_macro"],
        met_mlp_face_drop["f1_macro"],       met_mlp_face_nodrop["f1_macro"],
        met_cnn_face_bn["f1_macro"],         met_cnn_face_nobn["f1_macro"],
        met_ft_face_warm["f1_macro"],        met_ft_face_nowarm["f1_macro"],
        met_ft_face_resnet["f1_macro"],      met_ft_face_mobilenet["f1_macro"],
    ],
    "F1-micro": [
        met_mlp_moles_drop["f1_micro"],      met_mlp_moles_nodrop["f1_micro"],
        met_cnn_moles_bn["f1_micro"],        met_cnn_moles_nobn["f1_micro"],
        met_ft_moles_warm["f1_micro"],       met_ft_moles_nowarm["f1_micro"],
        met_ft_moles_resnet["f1_micro"],     met_ft_moles_mobilenet["f1_micro"],
        met_mlp_face_drop["f1_micro"],       met_mlp_face_nodrop["f1_micro"],
        met_cnn_face_bn["f1_micro"],         met_cnn_face_nobn["f1_micro"],
        met_ft_face_warm["f1_micro"],        met_ft_face_nowarm["f1_micro"],
        met_ft_face_resnet["f1_micro"],      met_ft_face_mobilenet["f1_micro"],
    ],
}

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))